# Capstone: Repetition code: syndrome statistics and the logical error

**Quantum Computing Advanced · Module 6 · about 5 hours** · notebook version 2026-10-10

**The question of this experiment.** Does the three-qubit repetition code protect a stored bit better than a single unencoded qubit, how does its logical error grow with the number of syndrome rounds, and at which noise level does it stop helping?

You run the experiment on a **noisy simulator**, a simulator that adds the errors of a real quantum computer described by a **noise model**. You compare what you measure with what the noise model predicts, say how certain every result is, and say what the experiment cannot tell you. The capstone brings together the code, the mitigation and the application of Modules 2, 3 and 4, and you write more of the code yourself than in a lab.

**The plan.**

1. write the statistics functions every capstone uses (Step 1) and meet the capstone noise model (Step 2);
2. build the **memory experiment**: encode, several rounds of syndrome measurement, a final measurement of the data (Step 3);
3. read every shot: write `detection_events()` and `decode()` and measure the **logical error** (Step 4);
4. compare the code with an unencoded qubit for 1 to 5 rounds (Step 5);
5. look at the **syndrome statistics**: how often each round signals an error (Step 6);
6. use the code to **detect** instead of correct: discard shots that signalled an error (Step 7);
7. find the **break-even** noise level, where the code stops beating a single qubit (Step 8);
8. get your verification value (Step 9).

**How it is graded.** Your capstone quiz in Canvas (the one for this experiment, 20 points, pass at 16) asks for your verification value from the check cell (6 points), three results from this notebook and four questions about uncertainty, interpretation and limits (2 points each). Question 1 gives you three personal numbers: ROUNDS, P2 and READOUT. At the end you write a short **summary** of your experiment (required, not graded), which you submit in the Course Completion module.

**Saving your work.** JupyterLite keeps your changes in this browser, so you can stop and come back. To keep a copy elsewhere, use **File → Download**. Run each code cell with **Shift + Enter**, in order; if you come back later, run the cells from the top again.

## Step 0: start Python

Run the cell below. The first time, Python can take up to a minute to start in your browser, and Matplotlib takes a few more seconds to load.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import qsim
from qsim import QuantumCircuit, NoiseModel, ReadoutError, depolarizing_error, simulate_density_matrix

print("Ready. qsim", qsim.__version__, "| NumPy", np.__version__)

## Step 1: how certain is a result from shots?

Every number you measure in this capstone comes from shots, so every number has an uncertainty. Three formulas cover all of them.

**A fraction.** If a result has probability p and you take S shots, the fraction f of shots with that result has standard deviation

**σ = √( p (1 − p) / S )**.

**An expectation value of ±1 results**, such as ⟨Z⟩ or ⟨X X X⟩: each shot gives +1 or −1, and the average E has standard deviation

**σ = √( (1 − E²) / S )**.

This is twice the σ of the fraction of +1 results, because E = 2 f − 1. For E = 0.8 and S = 4,000 it is √(0.36 / 4000) = 0.0095.

**A combination of independent results.** Mitigation combines several measured values: an estimate Ê = c₁ E₁ + c₂ E₂ + ... from independent runs with uncertainties σ₁, σ₂, ... has

**σ = √( (c₁ σ₁)² + (c₂ σ₂)² + ... )**.

Uncertainties add **in quadrature**. The difference of two results is the special case c = (1, −1): σ_diff = √(σ₁² + σ₂²). Large coefficients, as in an extrapolation, make the combination much less certain than any single run: this is the price of mitigation you met in Module 3.

**The course rule: 3σ.** A measured value agrees with an expected value when they differ by at most 3σ. A difference larger than 3σ_diff is **significant**: shot noise alone would almost never produce it. A smaller difference may be real, but your data cannot show it. Two cautions: the rule treats the spread as normal (bell-shaped), which is good when many shots give each result; and when you compare many settings, chance alone makes one "significant" difference likely somewhere (with 20 comparisons, about 5%). Treat a single surprising difference among many as a question to test again with new shots.

**Your task:** write the three functions below. Every later step uses them.

- `fraction_sigma(p, shots)`: √(p (1 − p) / shots).
- `expectation_sigma(E, shots)`: √((1 − E²) / shots).
- `combined_sigma(coefficients, sigmas)`: √(Σ (cᵢ σᵢ)²) for two lists of the same length.

In [ ]:
def fraction_sigma(p, shots):
    """The standard deviation of a fraction measured with `shots` shots, when the true probability is p."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete fraction_sigma() first.")


def expectation_sigma(E, shots):
    """The standard deviation of the average of `shots` results of +1 or -1 with expectation value E."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete expectation_sigma() first.")


def combined_sigma(coefficients, sigmas):
    """The standard deviation of sum(c_i * E_i) for independent E_i with standard deviations sigma_i."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete combined_sigma() first.")

In [ ]:
print("fraction_sigma(0.0613, 4000) =", round(fraction_sigma(0.0613, 4000), 5), "  (expected 0.00379)")
print("expectation_sigma(0.8, 4000) =", round(expectation_sigma(0.8, 4000), 5), "  (expected 0.00949)")
print("combined_sigma([1, -1], [0.01, 0.02]) =", round(combined_sigma([1, -1], [0.01, 0.02]), 5), "  (expected 0.02236)")
print("combined_sigma([1.875, -1.25, 0.375], [0.012, 0.015, 0.02]) =",
      round(combined_sigma([1.875, -1.25, 0.375], [0.012, 0.015, 0.02]), 5), "  (expected 0.03023)")

## Step 2: the capstone noise model

The noise model of the capstone is the one you used in Level 2 Module 6, with one addition:

- after every **two-qubit gate**, a **depolarizing error** with parameter P2 (with probability P2 the two qubits are replaced by a completely random state);
- after every **one-qubit gate**, the same on one qubit with P2 / 10;
- an **idle error** on `id` gates, used only by the repetition-code capstone (a qubit that waits while others are measured);
- at **readout**, a 0 is recorded as 1 with probability READOUT and a 1 as 0 with probability 2 · READOUT.

The course values are **P2 = 0.01** and **READOUT = 0.02**, several times worse than IBM's best devices, so that the effects show clearly in a few thousand shots. Your capstone quiz gives you your own P2 and READOUT (0.005 to 0.030 each) for the check cell.

The prepared cell builds the model and three tools:

- `exact_probabilities(qc, p2, readout)`: the noise model's **exact** probability of every recorded result, from the density matrix (Module 1) and the readout confusion; no shots, no randomness. This is the expected value the 3σ rule compares with.
- `sample_counts(probs, shots, seed)`: random shots drawn from those probabilities. A run of the noisy simulator gives counts with exactly these statistics; drawing them from the exact probabilities is much faster in the browser, and the seed makes your run repeatable.
- `apply_readout(probs, readout)`: the readout errors alone, applied to probabilities.

This capstone measures in the middle of the circuit, so it adds two more prepared tools in Step 3: `record_distribution(qc, p2, readout, idle)`, the exact probability of every recorded key of a circuit with mid-circuit measurements (each measurement splits the density matrix into branches, as in Module 2), and `logical_error()`.

In [ ]:
ONE_QUBIT_GATES = ["h", "x", "y", "z", "rx", "ry", "rz", "p", "s", "sdg", "t", "tdg", "sx"]
TWO_QUBIT_GATES = ["cx", "cz", "cp", "rzz", "swap"]
P2_COURSE, READOUT_COURSE = 0.01, 0.02


def confusion(readout):
    """Prepared. Rows: the true bit (0, 1); columns: the recorded bit."""
    return np.array([[1 - readout, readout], [2 * readout, 1 - 2 * readout]])


def capstone_noise_model(p2=P2_COURSE, readout=READOUT_COURSE, idle=0.0):
    """Prepared: depolarizing error p2 after two-qubit gates, p2/10 after one-qubit gates, idle on id gates, readout."""
    nm = NoiseModel()
    if p2 > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(p2 / 10, 1), ONE_QUBIT_GATES)
        nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), TWO_QUBIT_GATES)
    if idle > 0:
        nm.add_all_qubit_quantum_error(depolarizing_error(idle, 1), ["id"])
    if readout > 0:
        nm.add_all_qubit_readout_error(ReadoutError(confusion(readout).tolist()))
    return nm


def apply_readout(probs, readout):
    """Prepared. probs over m recorded bits (index bit j = classical bit j); each bit is flipped by the confusion."""
    probs = np.asarray(probs, dtype=float)
    m = int(round(math.log2(len(probs))))
    c = confusion(readout)
    for j in range(m):
        t = probs.reshape(2 ** (m - 1 - j), 2, 2 ** j)
        probs = np.einsum("aib,ij->ajb", t, c).reshape(-1)
    return probs


def exact_probabilities(qc, p2=P2_COURSE, readout=READOUT_COURSE, idle=0.0):
    """Prepared: the noise model's exact probability of each recorded result, {"bits": p}, Qiskit's key order.
    qc must end with its measurements (each classical bit receives one qubit). No shots are taken."""
    measured = {inst.clbits[0]: inst.qubits[0] for inst in qc.data if inst.name == "measure"}
    body = qc.copy()
    body.remove_final_measurements()
    nm = capstone_noise_model(p2, 0.0, idle)
    rho = simulate_density_matrix(body, nm if not nm.is_ideal() else None)   # the exact mixed state after the noisy gates
    m = len(measured)
    probs = rho.probabilities(qargs=[measured[j] for j in range(m)])
    probs = apply_readout(probs, readout)                                       # then the readout errors
    return {format(i, f"0{m}b"): float(p) for i, p in enumerate(probs) if p > 1e-15}


def sample_counts(probs, shots, seed):
    """Prepared: `shots` random results drawn from exact probabilities, as a quantum computer gives them."""
    keys = sorted(probs)
    p = np.array([probs[k] for k in keys])
    draw = np.random.default_rng(seed).multinomial(shots, p / p.sum())
    return {k: int(c) for k, c in zip(keys, draw) if c > 0}


print(capstone_noise_model())

## Step 3: the memory experiment

A **memory experiment** stores one bit and checks how well it survives. It is how IBM, Google and others test their codes. Your circuit, `memory_circuit(rounds, bit)`, uses the layout of Module 2: data qubits 0, 1 and 2, ancillas 3 and 4, and 2 · rounds + 3 classical bits.

1. **Prepare and encode.** If `bit` is 1, apply X to qubit 0. Then CX(0, 1) and CX(0, 2): the logical 0 is |000⟩, the logical 1 is |111⟩.
2. **Each round r = 0, 1, ...:**
   - an `id` gate on each data qubit. The data qubits wait while the ancillas are measured, and the noise model puts the **idle error** (here equal to P2) on these id gates;
   - CX(0, 3), CX(1, 3), CX(1, 4), CX(2, 4), in this order: ancilla 3 collects the parity of qubits 0 and 1, ancilla 4 that of qubits 1 and 2 (without noise the order does not matter; with noise it changes the result slightly, and the check uses this order);
   - measure ancilla 3 into classical bit 2r and ancilla 4 into classical bit 2r + 1;
   - reset both ancillas, so the next round starts again from |0⟩.
3. **Finally** measure data qubit d into classical bit 2 · rounds + d.

There is no correction during the circuit: the decoder reads everything afterwards. (Correcting with `if_test` in every round, as in Module 2, gives the same logical error for this code, because a correction only flips a data qubit the final majority vote would flip anyway.)

**Your task:** write `memory_circuit(rounds, bit=0)`. The prepared cell after it defines `bare_circuit(rounds, bit)`, a single unencoded qubit with one idle error per round (one id gate), and the exact tools. This is a simple reference, not an equal-time or equal-hardware comparison: a real round of the code takes longer than one idle step, because of its CX gates, measurements and resets.

In [ ]:
def memory_circuit(rounds, bit=0):
    """The memory experiment: encode `bit`, `rounds` rounds of syndrome measurement, then measure the data."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete memory_circuit() first.")

In [ ]:
def bare_circuit(rounds, bit=0):
    """Prepared: an unencoded qubit with one idle error per round (one id gate), then a measurement."""
    qc = QuantumCircuit(1, 1)
    if bit:
        qc.x(0)
    for _ in range(rounds):
        qc.id(0)
    qc.measure(0, 0)
    return qc


def record_distribution(qc, p2=P2_COURSE, readout=READOUT_COURSE, idle=None):
    """Prepared: the exact probability of every recorded key of a circuit with mid-circuit measurements.
    idle defaults to p2. Each measurement in the middle splits the state into branches (the course simulator's
    branch simulation, as for if_test in Module 2); the final measurements are read from each branch."""
    idle = p2 if idle is None else idle
    final = qc._final_measure_indices()
    meas = [qc.data[k] for k in sorted(final)]
    body = qc.copy_empty_like()
    body.data = [inst for k, inst in enumerate(qc.data) if k not in final]
    nm = capstone_noise_model(p2, readout, idle)
    branches = qsim._branch_states(body, nm if not nm.is_ideal() else None)
    qubits, clbits = [m.qubits[0] for m in meas], [m.clbits[0] for m in meas]
    out = {}
    for r, bits in branches:
        w = float(np.real(np.trace(r)))
        if w < 1e-15:
            continue
        probs = np.asarray(qsim.DensityMatrix(r / w).probabilities(qargs=qubits), dtype=float) * w
        probs = apply_readout(probs, readout)
        for o, p in enumerate(probs):
            if p < 1e-16:
                continue
            b = list(bits)
            for j, c in enumerate(clbits):
                b[c] = (o >> j) & 1
            key = "".join(str(x) for x in reversed(b))
            out[key] = out.get(key, 0.0) + float(p)
    return out


qc = memory_circuit(2)
print(qc.draw())
ops = [i.name for i in qc.data]
print("2 rounds:", ops.count("cx"), "CX,", ops.count("id"), "id,", ops.count("measure"), "measurements,", ops.count("reset"), "resets")
ideal = record_distribution(memory_circuit(3, 0), p2=0, readout=0)
print("Without noise, logical 0 after 3 rounds gives:", ideal, "  (expected {'000000000': 1.0})")

## Step 4: read every shot

Each shot gives a key of 2 · rounds + 3 bits, in Qiskit's order: classical bit 0 is the **rightmost** character. For 2 rounds, the key `"010" + "00" + "01"` means: round 0 syndrome (s₁, s₂) = (1, 0) (bits 0 and 1), round 1 syndrome (0, 0) (bits 2 and 3), final data bits d₀ d₁ d₂ = 0, 1, 0 (bits 4, 5, 6).

**Detection events.** A syndrome of (1, 0) can mean a data error that happened in this round, or one from an earlier round that is still there. What signals a *new* error is a **change**: the syndrome of round r XOR the syndrome of round r − 1 (before the first round, the syndrome counts as (0, 0)). A change is a **detection event**, the quantity real decoders work with (you met it in Module 2's saved device run). After the last round, the final data bits give one more syndrome, (d₀ XOR d₁, d₁ XOR d₂), compared with the last measured one.

**Decoding.** The simplest decoder is the majority of the three final data bits: it corrects any single flipped data qubit.

**Your task:**

- `detection_events(key, rounds)`: a list of rounds + 1 pairs (e₁, e₂), the changes of the syndrome, the last one from the final data bits.
- `decode(key, rounds)`: 0 or 1, the majority of the final data bits.

The prepared `logical_error(probs, rounds, bit)` then adds up the probability (or the fraction of shots) where `decode` disagrees with the stored bit.

In [ ]:
def detection_events(key, rounds):
    """rounds + 1 pairs: each round's syndrome XOR the previous one, then the syndrome of the final data bits."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete detection_events() first.")


def decode(key, rounds):
    """The logical bit: the majority of the three final data bits."""
    # YOUR CODE HERE
    raise NotImplementedError("Complete decode() first.")

In [ ]:
def logical_error(probs, rounds, bit=0):
    """Prepared: the probability (or fraction of shots) in which decode() disagrees with the stored bit."""
    total = sum(probs.values())
    return sum(p for k, p in probs.items() if decode(k, rounds) != bit) / total


print(detection_events("010" + "00" + "01", 2), "  (expected [(1, 0), (1, 0), (1, 1)])")
print(decode("010" + "00" + "01", 2), "  (expected 0)")

SHOTS, ROUNDS_DEMO = 4000, 3
exact3 = record_distribution(memory_circuit(ROUNDS_DEMO))
counts = sample_counts(exact3, SHOTS, seed=1)
p_exact = logical_error(exact3, ROUNDS_DEMO)
f = logical_error(counts, ROUNDS_DEMO)
s = fraction_sigma(p_exact, SHOTS)
print(f"Logical 0, {ROUNDS_DEMO} rounds, P2 = {P2_COURSE}, READOUT = {READOUT_COURSE}:")
print(f"   exact logical error {p_exact:.4f};  your run of {SHOTS} shots: {f:.4f} +- {s:.4f} ({(f - p_exact) / s:+.1f} sigma)")

**What to notice.** The exact logical error after 3 rounds is **0.0152**. A run of 4,000 shots lands within a few σ of it (σ ≈ 0.0019); the 3σ rule says whether a measured value agrees with the noise model. A small error rate needs many shots to measure precisely: with p ≈ 0.015, σ / p is about 13% at 4,000 shots.

## Step 5: does the code beat a single qubit?

An unencoded qubit stores a bit too. Over the same number of rounds it collects idle errors, and its readout error is not corrected. The code costs four noisy CNOTs per round and more qubits that can fail, but it corrects any single flipped data qubit.

Run the cell: it compares the exact logical error of the code with the exact error of a bare qubit (`bare_circuit`) for 1 to 5 rounds, and uses `combined_sigma` to say whether 4,000 shots of each could tell them apart.

In [ ]:
print(f"{'rounds':>7s}{'code':>10s}{'bare qubit':>12s}{'difference':>12s}{'sigma_diff':>12s}{'significant?':>14s}")
enc, bare = [], []
for r in range(1, 6):
    e = logical_error(record_distribution(memory_circuit(r)), r)
    b = record_distribution(bare_circuit(r)).get("1", 0.0)
    sd = combined_sigma([1, -1], [fraction_sigma(e, SHOTS), fraction_sigma(b, SHOTS)])
    enc.append(e); bare.append(b)
    print(f"{r:7d}{e:10.4f}{b:12.4f}{b - e:12.4f}{sd:12.4f}{('yes' if abs(b - e) > 3 * sd else 'no'):>14s}")

plt.figure(figsize=(6, 3.6))
plt.plot(range(1, 6), enc, "o-", label="repetition code")
plt.plot(range(1, 6), bare, "s--", label="bare qubit")
plt.xlabel("rounds"); plt.ylabel("logical error"); plt.legend(); plt.title("Stored 0, P2 = 0.01, READOUT = 0.02")
plt.tight_layout(); plt.show()

**What to notice.** The code's logical error (0.0102 after one round, 0.0152 after three) is less than half the bare qubit's (0.0247 and 0.0340), and with 4,000 shots of each the difference is significant at every number of rounds. Both errors grow with the rounds, because errors keep arriving; the code's grows because two flips in the same block, at any time before the final measurement, beat the majority vote. Part of the code's advantage comes from the readout: the majority vote also corrects one misread data bit.

## Step 6: syndrome statistics

How often does a round signal an error? Run the cell: for 3 rounds it gives the exact fraction of shots with a detection event in each round (either of the two syndrome bits changed), and the same from your 4,000-shot run with its σ.

In [ ]:
exact_rate = [0.0] * 4
run_rate = [0] * 4
for k, p in exact3.items():
    for i, e in enumerate(detection_events(k, 3)):
        if tuple(e) != (0, 0):
            exact_rate[i] += p
for k, c in counts.items():
    for i, e in enumerate(detection_events(k, 3)):
        if tuple(e) != (0, 0):
            run_rate[i] += c
labels = ["round 1", "round 2", "round 3", "final data"]
for lab, pe, k in zip(labels, exact_rate, run_rate):
    f = k / SHOTS
    print(f"{lab:>11s}: exact {pe:.4f}   your run {f:.4f} +- {fraction_sigma(pe, SHOTS):.4f}")

**What to notice.** About 13% of shots show a detection event in round 2 (exact **0.1292**), far more than the logical error: most detected errors are corrected or come from the ancillas and their readout, not from the stored bit. Round 1 is lower (0.0849) because its syndrome is compared with a perfect (0, 0), while every later round compares two noisy syndromes, so a single wrong syndrome makes two events, one when it appears and one when it goes away. The comparison with the final data (0.1187) involves the data readout instead of the ancilla readout. In a real device, the pattern of detection events in space and time is what a decoder such as minimum-weight matching works with.

## Step 7: detect instead of correct

A code that corrects one error can **detect** more. Instead of trusting the majority vote, you can throw away every shot that showed any detection event and keep only the "quiet" ones. This is **post-selection**. It lowers the error of the shots you keep, at the price of the shots you discard; it works for a memory test, but a long computation cannot simply be restarted every time an error appears.

Run the cell: it compares the logical error of all shots with that of the kept shots, exact and from your run.

In [ ]:
def quiet(k, rounds):
    return all(tuple(e) == (0, 0) for e in detection_events(k, rounds))

kept_p = sum(p for k, p in exact3.items() if quiet(k, 3))
kept_err = sum(p for k, p in exact3.items() if quiet(k, 3) and decode(k, 3) != 0) / kept_p
kept_shots = sum(c for k, c in counts.items() if quiet(k, 3))
kept_wrong = sum(c for k, c in counts.items() if quiet(k, 3) and decode(k, 3) != 0)
print(f"exact: all shots {logical_error(exact3, 3):.4f};  kept {kept_p:.1%} of shots, their logical error {kept_err:.5f}")
print(f"your run: kept {kept_shots} of {SHOTS} shots, {kept_wrong} of them wrong "
      f"({kept_wrong / kept_shots:.5f} +- {fraction_sigma(kept_err, kept_shots):.5f})")

**What to notice.** Post-selection keeps about **72%** of the shots, and among them the logical error falls to about **0.0024**, roughly six times lower. With 4,000 shots only a handful of kept shots are wrong, so the measured value has a large relative uncertainty: small error rates need many shots. Detection is cheap and powerful for experiments that can be repeated, which is why early demonstrations of codes often report post-selected results; correction is what a long computation needs.

## Step 8: break-even

The code helps only if its extra gates add fewer errors than it removes. Run the cell: it scans P2 (and the idle error with it) from 0.005 to 0.08 for 3 rounds, with READOUT = 0.02, and finds the first P2 where the code is worse than a bare qubit.

In [ ]:
p2_values = [round(0.005 * i, 3) for i in range(1, 17)]
code_err = [logical_error(record_distribution(memory_circuit(3), p2=p), 3) for p in p2_values]
bare_err = [record_distribution(bare_circuit(3), p2=p).get("1", 0.0) for p in p2_values]
for p, e, b in zip(p2_values, code_err, bare_err):
    print(f"P2 = {p:.3f}:  code {e:.4f}   bare {b:.4f}   {'code better' if e < b else 'bare better'}")

fine = [round(0.001 * i, 3) for i in range(5, 81)]
cross = next(p for p in fine if logical_error(record_distribution(memory_circuit(3), p2=p), 3)
             > record_distribution(bare_circuit(3), p2=p).get("1", 0.0))
print("\nBreak-even (first P2 where the code is worse, steps of 0.001):", cross)
plt.figure(figsize=(6, 3.6))
plt.plot(p2_values, code_err, "o-", label="repetition code, 3 rounds")
plt.plot(p2_values, bare_err, "s--", label="bare qubit")
plt.axvline(cross, color="grey", ls=":")
plt.xlabel("P2 (and idle error)"); plt.ylabel("logical error"); plt.legend(); plt.tight_layout(); plt.show()

**What to notice.** The code wins below P2 ≈ **0.037** and loses above it. This break-even belongs to this toy model and this comparison (one idle error per round for the bare qubit, the same depolarizing error on every gate): it is not a fault-tolerance threshold, and this experiment does not test whether longer codes would do better. It shows the idea behind a threshold: encoding helps only when the extra gates add fewer errors than the code removes.

## Step 9: your personal check

Open your capstone quiz in Canvas. Question 1 shows your own numbers: ROUNDS (1 to 5), P2 and READOUT (0.005 to 0.030). Type them below and run the next two cells. The check cell tests the three statistics functions of Step 1 and every function you wrote for this capstone. Only if every test passes does it print your **verification value**: 100,000 times the exact logical error of a stored 0 after ROUNDS rounds with your noise model (idle error equal to P2), decoded by majority vote, rounded to a whole number. For example, ROUNDS, P2, READOUT = 3, 0.01, 0.02 gives 1516.

In [ ]:
ROUNDS = 0    # your number from Canvas
P2 = 0    # your number from Canvas
READOUT = 0    # your number from Canvas

In [ ]:
# The course's check code is in l3_m6_common.py and l3_m6_rep_check.py, next to this notebook.
# It is kept out of the notebook so that the course's reference solutions are not on screen.
# You may open the file to see how your functions are tested, but write your own functions first.
from l3_m6_rep_check import check_l3_m6_rep

passed, messages, value = check_l3_m6_rep(fraction_sigma, expectation_sigma, combined_sigma, memory_circuit, detection_events, decode,
                                          ROUNDS, P2, READOUT)
for m in messages:
    print(m)
if passed:
    print("\nAll tests passed. Your verification value is", value)
else:
    print("\nNot yet: fix the item above and run this cell again.")

## Limits of this experiment

- **Only one kind of error is tested.** A stored 0 or 1 suffers only from bit flips. The noise model also makes phase flips, which this code cannot see; a superposition such as (|000⟩ + |111⟩)/√2 would lose its phase, as you saw in Module 2. A real memory must protect both, which needs a larger code such as the surface code.
- **The decoder is simple.** The majority vote ignores the syndrome history. A decoder that uses the detection events in time (for example minimum-weight matching) is better at telling a likely measurement error from a likely data error.
- **The comparison is a toy one.** The bare qubit gets one idle error per round, while a round of the code also has CX gates, measurements and resets that take time on hardware. The comparison is not equal in time or hardware, and the break-even of Step 8 holds for this model only.
- **The noise model is simple.** Errors are independent and the same for every qubit and gate; real devices have crosstalk, leakage, drifting calibrations and errors that hit several qubits at once (and readout that disturbs neighbouring qubits).
- **Exact values, sampled runs.** Your 4,000-shot run measures each rate with an uncertainty; small rates, such as the post-selected error, need far more shots for a precise value.
- **No connectivity.** Here any qubit can share a CX with any other. On a heavy-hex chip the ancillas must sit next to the data qubits, which IBM's layouts allow for this code, but larger codes need careful layouts.

These limits are not failures of your experiment: every experiment has them. Saying what a result does and does not show is part of reporting it.

## Optional: run it on a real IBM quantum computer

If you have an IBM Quantum account with an Open Plan instance (see **Set Up Your Tools** in Start Here), the [Module 6 hardware notebook in Colab](https://colab.research.google.com/github/CSU-Physics/quantum-computing-notebooks/blob/main/colab/level3/QC-L3-M6-hardware-colab.ipynb) runs this capstone's reference circuits on a real computer and compares them with the ideal values and with this noise model. Your API key stays in Colab's **Secrets**; never type it into a cell. The Open Plan gives up to 10 minutes of quantum computer time every 28 days; the hardware notebook uses well under a minute of it, but the queue can take from seconds to hours. This step is not graded, and the badge does not depend on it.

## Your capstone summary (required, not graded)

Write five short answers here (double-click this cell to edit it). They are how a result is reported in a lab notebook or a paper, and they are good preparation for the capstone quiz. Then paste them into **Your Capstone Summary** in the Course Completion module in Canvas: it is not graded, but it is required for the badge.

1. **Question.** What did you want to find out?
2. **Method.** Which circuits, which noise model (P2, READOUT), how many shots, which mitigation or decoding?
3. **Result.** Your main numbers, each with its uncertainty (for example 0.976 ± 0.010).
4. **Interpretation.** What do the numbers say about the question? Which differences are significant (more than 3σ)?
5. **Limits.** What does this experiment not show?

**Next in Canvas:** the **Capstone Quiz — Repetition Code**, then the Module 6 time log.